# Inference v4 - scratch / r34 / ensemble, flip-TTA, per-class post-processing, optional classification gate

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import cv2

IMG_H, IMG_W = 128, 800
DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]
BS = 16
R34_NAME = "best_r34.pt"

CONFIG = dict(
    models=["r34"],
    tta={"r34": True},
    weights=[1.0],
    gate_source="cls",
    tau=0.5,
    min_area=[100, 50, 200, 200],
    gate=[0.1, 0.2, 0.2, 0.2]
)

def find_file(name, known=None):
    if known and os.path.exists(known):
        return known
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found - attach the right input to this notebook")
    return sorted(hits)[0]

def find_image_dir(sample_id):
    hits = glob.glob(f"/kaggle/input/**/{sample_id}.*", recursive=True)
    if not hits:
        raise FileNotFoundError(f"image {sample_id} not found - attach the competition data")
    return os.path.dirname(hits[0])

SAMPLE_CSV = find_file("sample_submission.csv", "/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv")
CKPTS = {}
if "scratch" in CONFIG["models"]:
    CKPTS["scratch"] = find_file("best_unet_scratch.pt", "/kaggle/input/notebooks/mruniksurya/milestone-2/best_unet_scratch.pt")
if "r34" in CONFIG["models"]:
    CKPTS["r34"] = find_file(R34_NAME, f"/kaggle/input/notebooks/mruniksurya/milestone-3-r34/{R34_NAME}")
assert CONFIG["gate_source"] != "cls" or "r34" in CONFIG["models"], "the classification gate needs the r34 model"
print("CONFIG:", CONFIG); print("checkpoints:", CKPTS)

## 1. RLE encoder and post-processing

In [ ]:
def rle_decode(rle, shape=(IMG_H, IMG_W)):
    """1-based, column-major RLE string -> binary (H, W) mask. NaN / blank -> all zeros."""
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if isinstance(rle, str) and rle.strip():
        tok = np.array(rle.split(), dtype=np.int64)
        starts = tok[0::2] - 1
        ends = starts + tok[1::2]
        for s, e in zip(starts, ends):
            flat[max(s, 0):min(e, flat.size)] = 1
    return flat.reshape(shape, order="F")

def rle_encode(mask):
    """Binary (H, W) mask -> RLE string (1-based, column-major). Empty mask -> empty string."""
    flat = np.asarray(mask).flatten(order="F")
    padded = np.concatenate([[0], flat, [0]])
    k = np.where(padded[1:] != padded[:-1])[0] + 1
    return " ".join(f"{s} {e - s}" for s, e in zip(k[0::2], k[1::2]))

In [ ]:
import cv2

def postprocess(prob, tau, min_area, gate, gate_score=None):
    """prob (H,W) of ONE (image, class) pair -> binary uint8 mask.
    gate_score: the value compared with the gate: the classification probability, or (default) the most confident pixel.
    If it is below the gate the whole mask is emptied. min_area: regions (8-connectivity) smaller than this are removed."""
    score = prob.max() if gate_score is None else gate_score
    if score < gate:
        return np.zeros(prob.shape, dtype=np.uint8)
    mask = (prob > tau).astype(np.uint8)
    if min_area > 0 and mask.any():
        n, lab, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        keep = np.zeros(n, dtype=bool)
        keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_area
        mask = keep[lab].astype(np.uint8)
    return mask

## 2. Models and preprocessing (identical to the training notebooks and the analysis notebook)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34

class DoubleConv(nn.Module):
    """(Conv3x3 -> BatchNorm -> ReLU) x 2 : the basic U-Net building block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.block(x)

class UNetScratch(nn.Module):
    """Milestone-2 scratch model: 4 encoder levels, 1 input channel, 4 logits. 7,762,564 parameters."""
    def __init__(self, in_ch=1, n_classes=4, base=32):
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]
        self.enc1, self.enc2 = DoubleConv(in_ch, c[0]), DoubleConv(c[0], c[1])
        self.enc3, self.enc4 = DoubleConv(c[1], c[2]), DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c[3], c[4])
        self.up4, self.dec4 = nn.ConvTranspose2d(c[4], c[3], 2, stride=2), DoubleConv(c[3] * 2, c[3])
        self.up3, self.dec3 = nn.ConvTranspose2d(c[3], c[2], 2, stride=2), DoubleConv(c[2] * 2, c[2])
        self.up2, self.dec2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2), DoubleConv(c[1] * 2, c[1])
        self.up1, self.dec1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2), DoubleConv(c[0] * 2, c[0])
        self.head = nn.Conv2d(c[0], n_classes, kernel_size=1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)

class UpBlock(nn.Module):
    """Upsample to the skip's size, concatenate, DoubleConv."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
        return self.conv(torch.cat([x, skip], dim=1))

class ResNetUNet(nn.Module):
    """Milestone-3 model: ResNet34 encoder (1 gray channel) + U-Net decoder + image-level classification head.
    forward(x) -> (segmentation logits (B,4,H,W), classification logits (B,4)). Identical to the training notebook."""
    def __init__(self, base, n_classes=4):
        super().__init__()
        w = base.conv1.weight.data.sum(dim=1, keepdim=True)
        conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        conv1.weight.data.copy_(w)
        self.stem = nn.Sequential(conv1, base.bn1, base.relu)
        self.pool = base.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = base.layer1, base.layer2, base.layer3, base.layer4
        self.up3 = UpBlock(512, 256, 256)
        self.up2 = UpBlock(256, 128, 128)
        self.up1 = UpBlock(128, 64, 64)
        self.up0 = UpBlock(64, 64, 32)
        self.final = DoubleConv(32, 16)
        self.head = nn.Conv2d(16, n_classes, kernel_size=1)
        self.cls_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(512, n_classes))
    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)
        d = self.up3(e4, e3)
        d = self.up2(d, e2)
        d = self.up1(d, e1)
        d = self.up0(d, e0)
        d = F.interpolate(d, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(self.final(d)), self.cls_head(e4)

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

IMG_MEAN, IMG_STD = 0.3314, 0.1892      # training-set statistics shared by the scratch and r34 models

def to_input(u8):
    """u8: (B,H,W) uint8 gray -> (B,1,H,W) float tensor, (pixel/255 - mean)/std. Used by BOTH models."""
    x = torch.from_numpy(np.ascontiguousarray(u8)).float().div_(255.0)
    return ((x - IMG_MEAN) / IMG_STD).unsqueeze(1)

@torch.no_grad()
def predict(model, u8, tta=False):
    """Returns (probabilities (B,4,H,W), classification probabilities (B,4) or None) on the GPU.
    tta=True averages the plain image with its horizontal and vertical flip."""
    x = to_input(u8).to(DEVICE)
    def fwd(inp):
        out = model(inp)
        if isinstance(out, tuple):
            return torch.sigmoid(out[0]), torch.sigmoid(out[1])
        return torch.sigmoid(out), None
    p, c = fwd(x)
    if tta:
        for dim in (-1, -2):
            p2, c2 = fwd(x.flip(dim))
            p = p + p2.flip(dim)
            if c is not None:
                c = c + c2
        p = p / 3.0
        c = None if c is None else c / 3.0
    return p, c

def load_model(kind, ckpt):
    if kind == "scratch":
        m = UNetScratch(in_ch=1, n_classes=4, base=32)
    elif kind == "r34":
        m = ResNetUNet(resnet34(weights=None))          # architecture only; the weights come from the checkpoint
    else:
        raise ValueError(kind)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return m.to(DEVICE).eval()

In [ ]:
models = {kind: load_model(kind, path) for kind, path in CKPTS.items()}
for kind, m in models.items():
    print(kind, f"{sum(p.numel() for p in m.parameters()):,} parameters")

## 4. Predict the test set

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
test_ids = sample["image_id"].drop_duplicates().tolist()
img_dir = find_image_dir(test_ids[0])
files = {os.path.splitext(f)[0]: f for f in os.listdir(img_dir)}
print("test images:", len(test_ids), "| folder:", img_dir)

w = np.asarray(CONFIG["weights"], dtype=np.float32); w = w / w.sum()
rle_lookup = {}
for s in range(0, len(test_ids), BS):
    ids = test_ids[s:s + BS]
    u8 = np.stack([cv2.imread(os.path.join(img_dir, files[i]), cv2.IMREAD_GRAYSCALE) for i in ids])
    assert u8.shape[1:] == (IMG_H, IMG_W), u8.shape
    p_sum, cls_prob = None, None
    for kind, wk in zip(CONFIG["models"], w):
        p, c = predict(models[kind], u8, CONFIG["tta"].get(kind, False))
        p_sum = p * float(wk) if p_sum is None else p_sum + p * float(wk)
        if kind == "r34":
            cls_prob = c.float().cpu().numpy()
    probs = p_sum.float().cpu().numpy()                                   # (B,4,H,W)
    for b, iid in enumerate(ids):
        for k, d in enumerate(DEFECTS):                                   # channel order a,b,c,d as in training
            gs = cls_prob[b, k] if CONFIG["gate_source"] == "cls" else None
            m = postprocess(probs[b, k], CONFIG["tau"], CONFIG["min_area"][k], CONFIG["gate"][k], gs)
            r = rle_encode(m)
            rle_lookup[(iid, d)] = r if r else " "
print("predicted pairs:", len(rle_lookup))

## 5. submission and sanity-check

In [ ]:
sub = sample.copy()
sub["mask_rle"] = [rle_lookup[(i, d)] for i, d in zip(sub["image_id"], sub["defect_type"])]
sub.to_csv("/kaggle/working/submission.csv", index=False)

assert len(sub) == len(sample) == 12000 and list(sub.columns) == list(sample.columns)
empty = sub["mask_rle"].str.strip().eq("")
print("rows:", len(sub), "| overall empty rate:", round(float(empty.mean()), 4))
print("empty rate per class:"); print(empty.groupby(sub["defect_type"]).mean().round(3).to_string())
print("reference: train true empty rates a 0.905, b 0.973, c 0.64, d 0.85; the 0.56021 submission (scratch + TTA + gates) predicted about a 0.69, b 0.96, c 0.40, d 0.74 empty on validation")
sub.head()